# 第11課 - 代理對代理 (A2A) 協議


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 什麼是 A2A 協議？

The **Agent-to-Agent (A2A) 協議**是一個開放標準，使 AI 代理人能夠溝通，
互相發現並協作 — 即使它們建立於不同的框架或由不同的服務所
託管。

主要概念：

- **Discovery** – 代理人會發佈 *代理人卡*，描述其能力，使其他代理人(或編排器)能
  更容易找到適合處理任務的專家。
- **Message Passing** – 代理人透過共同的協議交換結構化訊息，因此來自一個代理人的
  請求可以被另一個代理人理解並完成，而不受其內部
  實作的限制。
- **Task Lifecycle** – A2A 定義了像是 *已提交*、*處理中*、*已完成* 和
  *失敗* 等狀態，讓編排器能完整掌握委派任務的進度。

在本課中，我們模擬 A2A 風格的協作，將三個專精於旅遊的代理人串接成一個工作流程，讓每個代理人貢獻其專長並將結果傳遞給下一位。


## 建立專門的旅行代理人


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 透過工作流程的多代理協作

我們將三個代理連接成一個串行的工作流程，模仿 A2A 訊息傳遞：

1. **CurrencyExchangeAgent** 接收使用者請求並產出貨幣指引。
2. **ActivityPlannerAgent** 接收已豐富的上下文並加入活動建議。
3. **TravelManagerAgent** 將兩者輸入綜合成最終的旅遊簡報。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 在生產環境中理解 A2A

在生產環境中，A2A 協議可啟用強大的跨服務場景：

| 能力 | 說明 |
|---|---|
| **跨框架互操作性** | 使用一個框架所構建的代理可以將任務委派給使用任何其他符合 A2A 的框架所構建的代理，從而實現真正的跨組織互操作性。 |
| **服務邊界** | 代理可以位於不同的微服務、雲區域，甚至不同的組織，同時仍能無縫協作。 |
| **動態發現** | 編排器可以在運行時查詢 Agent Card 註冊表，以尋找最適合處理某個子任務的專家。 |
| **串流與推送通知** | A2A 支援 Server-Sent Events (SSE) 以提供實時進度更新，以及針對長時間運行任務的推送通知。 |

我們上面構建的工作流程是此模式的一個簡化、同一進程內的版本。在實際
部署中，每個代理會公開一個 HTTP 端點、發佈 Agent Card，並進行通訊
透過 A2A JSON-RPC 協議。


## 摘要

在本課程中您學到：

1. **何謂 A2A 協議** — 一種用於代理人之間發現、訊息傳遞，
   以及任務管理的開放標準。
2. **如何建立專門化代理人** — 貨幣兌換代理人、活動策劃代理人，以及旅遊管理協調器。
3. **如何將代理人串接到工作流程** — 使用 `WorkflowBuilder` 來建模序列式的
   訊息傳遞。
4. **A2A 在生產環境如何運作** — 促成跨框架、跨服務的協作，
   並具備動態發現與串流更新。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件已使用 AI 翻譯服務 Co-op Translator（https://github.com/Azure/co-op-translator）進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始語言版本之文件應被視為具權威性的依據。對於關鍵資訊，建議採用專業人工翻譯。對因使用本翻譯而引致之任何誤解或誤釋，我們概不負任何責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
